# 85 A judge that reads the results (A100)

The judge of notebook 80 (`judge_v1`) sees only the SQL text of a candidate. This notebook trains **`judge_v2`**, which also sees **what the query returned** when run with the answer-time cap: the number of rows and the first 5 rows, or the error ("does not compile", "still running after 1 s", ...).

Everything else is the same as for `judge_v1`: the same training data (`judge_data_v1`, train questions only), the same settings, and the same starting point (a copy of the generator's adapter, **not** `judge_v1`). So the two judges differ only in what they read. `judge_v2` **replaces** `judge_v1`; it is not trained on top of it. The final system stays one frozen 7B with two adapters: generator and judge.

Steps:
1. **Stage the train databases and run the training candidates** once each with the 1 s cap (about 15 minutes; the validation part was already run).
2. **Train `judge_v2`** (about 1 h 45 min).
3. **Validation** on the held-out databases, next to `judge_v1`.
4. **Score** the candidates of validation, dev and mini-dev (about 20 minutes).
5. **Time the answer path** with `judge_v2` (about 15 minutes): it now waits for the query results before judging.

About **2.5 hours, roughly 25 to 30 units**. Everything resumes after a disconnect: run the setup and config cells, then the cell that was running.

**Runtime > Change runtime type > A100.** Reopen this notebook from the GitHub link after every code update.

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = True  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
# Training libraries (transformers and peft, pinned in pyproject.toml); vLLM above brings torch.
!pip install -q -e "{REPO_DIR}[train]"
# Colab ships torchao 0.10.0, which peft rejects with an error (it needs 0.16 or newer). We do not use it.
!pip uninstall -q -y torchao
import importlib.metadata as md
print({p: md.version(p) for p in ("torch", "transformers", "peft", "vllm")})

In [ ]:
import json, os
R = f"{DRIVE_ROOT}/effsql/runs"
DATA = "judge_data_v1"    # the judges' training data (train questions only)
JUDGE1 = "judge_v1"       # the judge that reads SQL only (notebook 80)
JUDGE = "judge_v2"        # the judge that also reads results (this notebook)
CAND = "judge_cand_v1"    # candidates, labels and answer-time results for val, dev and mini-dev
assert os.path.exists(f"{R}/{DATA}/data_meta.json"), f"{DATA} is missing on Drive"
for s in ("val", "dev", "minidev"):
    assert os.path.exists(f"{R}/{CAND}/exec_{s}.jsonl"), f"exec_{s}.jsonl is missing: run 83_vote first"
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("CPU cores:", os.cpu_count())

### 1. Stage the train databases (30 GB, a few minutes) and run the training candidates

In [ ]:
!effsql stage --split train
!python -m effsql.judge exec-data --run {DATA}

### 2. Train `judge_v2` (about 1 h 45 min)

In [ ]:
!python -m effsql.judge train --run {JUDGE} --data {DATA} --with-results

### 3. Validation on the held-out databases, next to `judge_v1`

`auc`: how well p separates correct from wrong candidates (0.5 is chance, 1 is perfect). `top1`: among questions with at least one correct candidate, how often the highest-p candidate is correct.

In [ ]:
for j in (JUDGE1, JUDGE):
    v = json.load(open(f"{R}/{j}/judge_meta.json"))["val_final"]
    print(j, {k: round(v[k], 3) for k in ("logloss", "accuracy", "auc", "top1")})

### 4. Score the candidates of validation, dev and mini-dev

In [ ]:
!python -m effsql.judge score --run {CAND} --judge {JUDGE} --splits val,dev,minidev

### 5. Time the answer path with `judge_v2` (100 dev questions, one at a time)

In [ ]:
!effsql stage --split dev
!python -m effsql.judge latency --judge {JUDGE} --out latency_v2 --split dev

### Done: disconnect now, then run `86_judge_results_report` on a CPU runtime

*Runtime > Disconnect and delete runtime.* An idle GPU keeps using compute units.